# 2. Branches

A branch is a name pointing at a commit. It lets you work on two things at
once without mixing them up. In this chapter we build this:

```mermaid
gitGraph
    commit id: "volume"
    branch froude
    commit id: "froude number"
    checkout main
    branch tpc
    commit id: "tonnes per cm"
```

In [1]:
import os, shutil, stat, sys
from pathlib import Path


def remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)  # git makes its object files read-only
    func(path)


work = Path.home() / "git-course" / "02-branches"
if work.exists():
    shutil.rmtree(work, onexc=remove_readonly)
work.mkdir(parents=True)
%cd {work}

/home/emircem/git-course/02-branches


## Starting point

A repository with one commit.

In [2]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

Writing hydrostatics.py


In [3]:
!git init
!git add hydrostatics.py
!git commit -m "Add volume and displacement for a box barge"

Initialized empty Git repository in /home/emircem/git-course/02-branches/.git/
[main (root-commit) 792ff00] Add volume and displacement for a box barge
 1 file changed, 13 insertions(+)
 create mode 100644 hydrostatics.py


## Listing branches

In [4]:
!git branch

* main


One branch, called `main`. The `*` marks the one you are on.

## Making a branch

We want to add the Froude number, but we are not sure we will keep it. A
branch is the place for that. `git switch -c` creates a branch and moves to
it:

In [5]:
!git switch -c froude

Switched to a new branch 'froude'


In [6]:
!git branch

* froude
  main


In [7]:
%%writefile -a hydrostatics.py


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5

Appending to hydrostatics.py


In [8]:
!git add hydrostatics.py
!git commit -m "Add the Froude number"

[froude d1d8a6f] Add the Froude number
 1 file changed, 5 insertions(+)


## Switching back

Now go back to `main` and look at the file:

In [9]:
!git switch main

Switched to branch 'main'


In [10]:
print(Path("hydrostatics.py").read_text())

"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)



`froude_number` is gone! Not deleted — it is safe on the `froude` branch.
Switching branches changes the files in your folder to match that branch.

## A second branch

Tonnes per centimetre: how much cargo sinks the barge by one centimetre.
We start it from `main`, next to `froude`:

In [11]:
!git switch -c tpc

Switched to a new branch 'tpc'


In [12]:
%%writefile -a hydrostatics.py


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0

Appending to hydrostatics.py


In [13]:
!git add hydrostatics.py
!git commit -m "Add tonnes per centimetre immersion"

[tpc 5838297] Add tonnes per centimetre immersion
 1 file changed, 5 insertions(+)


## Seeing the branches

This command draws the history as a graph. Remember it:

In [14]:
!git log --oneline --graph --all

* 5838297 (HEAD -> tpc) Add tonnes per centimetre immersion
| * d1d8a6f (froude) Add the Froude number
|/  
* 792ff00 (main) Add volume and displacement for a box barge


Two branches grew from the same commit. Neither knows about the other.

In [15]:
!git branch -v

  froude d1d8a6f Add the Froude number
  main   792ff00 Add volume and displacement for a box barge
* tpc    5838297 Add tonnes per centimetre immersion


## Deleting a branch

`git branch -d` deletes a branch, but refuses if that would lose work:

In [16]:
# This one fails on purpose: froude has a commit that main does not have.
!git branch -d froude

error: the branch 'froude' is not fully merged
hint: If you are sure you want to delete it, run 'git branch -D froude'
hint: Disable this message with "git config set advice.forceDeleteBranch false"


We keep both branches. Chapter 3 joins them back together.

## Summary

| I want to… | Command |
| - | - |
| list branches | `git branch` |
| create a branch and move to it | `git switch -c <name>` |
| move to a branch | `git switch <name>` |
| see all branches as a graph | `git log --oneline --graph --all` |
| delete a branch | `git branch -d <name>` |

Make a branch for every new piece of work, and give it a name that says
what it does.

## Exercises

1. Switch between `froude` and `tpc` and print the file each time.
2. Create a branch called `experiment`, then delete it with
   `git branch -d experiment`.

Next: [3. Merging](03-merging.ipynb)